# MITMA: exploración para Bronze y Silver
Perfil completo de los CSV descargados; `LIMIT` solo en las muestras. Trabajamos en memoria, sin modificar Bronze. Las conclusiones se limitan a las fechas disponibles.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'mitma').is_dir():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'mitma'
assert DATA.is_dir(), 'Abre desde project/ o project/notebooks/.'
con = duckdb.connect(':memory:')
con.execute("SET threads = 4")
con.execute("SET enable_progress_bar = false")

def leer(ruta, cabecera=True):
    return con.sql("SELECT * FROM read_csv(?, delim='|', header=?, all_varchar=true, "
                   "nullstr='__NULL_NO_PUBLICADO__')", params=[str(ruta), cabecera])

print('Datos:', DATA)
print('DuckDB:', duckdb.__version__)

Datos: /home/pablo/Desktop/muceim/Big Data Engineering/project/data/mitma
DuckDB: 2.0.0


## 1. Inventario
Los `.shp`, `.dbf`, `.shx`, etc. se conservan juntos: no son indicadores distintos. En este sprint perfilamos los CSV; no transformamos geometrías.

In [2]:
inventario = pd.DataFrame([
    {'archivo': str(p.relative_to(DATA)), 'MiB': round(p.stat().st_size / 2**20, 2)}
    for p in sorted(DATA.rglob('*')) if p.is_file()
])
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(inventario)

,archivo,MiB
0,LEEME_Especificaciones_indicadores_OpenDataMovilidad.xlsx,0.13
1,LICENCIA de datos abiertos del MITMA 20201203.pdf,0.15
2,calidad/diarios/20230101_distritos_descartados.csv,0.00
3,por-GAU/pernoctaciones/diarios/20230101_Pernoctaciones_GAU.csv.gz,2.43
4,por-GAU/personas/diarios/20230101_Personas_dia_GAU.csv.gz,0.42
5,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,72.78
6,por-distritos/pernoctaciones/diarios/20230101_Pernoctaciones_distritos.csv.gz,3.70
7,por-distritos/personas/diarios/20230101_Personas_dia_distritos.csv.gz,0.78
8,por-distritos/viajes/diarios/20230101_Viajes_distritos.csv.gz,149.26
9,por-municipios/pernoctaciones/diarios/20230101_Pernoctaciones_municipios.csv.gz,2.89


In [3]:
# Materializamos una vez para no descomprimir en cada consulta.
for familia in ['viajes', 'pernoctaciones', 'personas']:
    ruta = DATA / 'por-*' / familia / 'diarios' / '*.csv.gz'
    con.sql("SELECT * FROM read_csv(?, delim='|', header=true, all_varchar=true, "
            "nullstr='__NULL_NO_PUBLICADO__', filename=true)", params=[str(ruta)]).create_view('entrada')
    con.sql(f"""
        CREATE TEMP TABLE raw_{familia} AS
        SELECT regexp_extract(filename, 'por-([^/]+)/', 1) AS nivel,
               replace(filename, '{DATA}/', '') AS archivo_origen,
               * EXCLUDE (filename)
        FROM entrada
    """)

for familia in ['viajes', 'pernoctaciones', 'personas']:
    print(familia)
    display(con.sql(f"""
        SELECT nivel, archivo_origen, count(*) AS filas,
               min(fecha) AS fecha_min, max(fecha) AS fecha_max
        FROM raw_{familia} GROUP BY ALL ORDER BY nivel, archivo_origen
    """).df())
    display(con.sql(f'SELECT * FROM raw_{familia} LIMIT 3').df())

viajes


,nivel,archivo_origen,filas,fecha_min,fecha_max
0,GAU,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,6222637,20230101,20230101
1,distritos,por-distritos/viajes/diarios/20230101_Viajes_d...,14642142,20230101,20230101
2,municipios,por-municipios/viajes/diarios/20230101_Viajes_...,9693500,20230101,20230101


,nivel,archivo_origen,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
0,GAU,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,20230101,00,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.922,12.806
1,GAU,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,20230101,00,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,5.202,22.098
2,GAU,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,20230101,00,01001,01009_AM,0.5-2,frecuente,casa,no,no,01,>15,NA,NA,1.0,1.0


pernoctaciones


,nivel,archivo_origen,filas,fecha_min,fecha_max
0,GAU,por-GAU/pernoctaciones/diarios/20230101_Pernoc...,434829,20230101,20230101
1,distritos,por-distritos/pernoctaciones/diarios/20230101_...,726907,20230101,20230101
2,municipios,por-municipios/pernoctaciones/diarios/20230101...,560635,20230101,20230101


,nivel,archivo_origen,fecha,zona_residencia,zona_pernoctacion,personas
0,GAU,por-GAU/pernoctaciones/diarios/20230101_Pernoc...,20230101,01001,01001,2329.644
1,GAU,por-GAU/pernoctaciones/diarios/20230101_Pernoc...,20230101,01001,01017_AM,16.682
2,GAU,por-GAU/pernoctaciones/diarios/20230101_Pernoc...,20230101,01001,01028_AM,2.922


personas


,nivel,archivo_origen,filas,fecha_min,fecha_max
0,GAU,por-GAU/personas/diarios/20230101_Personas_dia...,64792,20230101,20230101
1,distritos,por-distritos/personas/diarios/20230101_Person...,119296,20230101,20230101
2,municipios,por-municipios/personas/diarios/20230101_Perso...,81805,20230101,20230101


,nivel,archivo_origen,fecha,zona_pernoctacion,edad,sexo,numero_viajes,personas
0,GAU,por-GAU/personas/diarios/20230101_Personas_dia...,20230101,01001,0-25,hombre,0,251.165
1,GAU,por-GAU/personas/diarios/20230101_Personas_dia...,20230101,01001,0-25,hombre,1,53.172
2,GAU,por-GAU/personas/diarios/20230101_Personas_dia...,20230101,01001,0-25,hombre,2,59.478


## 2. Estructura y calidad
Primero `SUMMARIZE` con los valores originales como texto. Después perfilamos las medidas convertidas. Los ceros iniciales, vacíos y `NA` no se reinterpretan al leer.

In [4]:
for familia in ['viajes', 'pernoctaciones', 'personas']:
    for nivel in ['distritos', 'municipios', 'GAU']:
        print(f'{familia} / {nivel}')
        display(con.sql(f"SUMMARIZE SELECT * EXCLUDE (nivel, archivo_origen) FROM raw_{familia} WHERE nivel = '{nivel}'").df())

viajes / distritos


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,14642142,0.0
1,periodo,VARCHAR,00,23,24,None,None,None,None,None,14642142,0.0
2,origen,VARCHAR,01001,externo,3895,None,None,None,None,None,14642142,0.0
3,destino,VARCHAR,01001,externo,3895,None,None,None,None,None,14642142,0.0
4,distancia,VARCHAR,0.5-2,>50,4,None,None,None,None,None,14642142,0.0
5,actividad_origen,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,14642142,0.0
6,actividad_destino,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,14642142,0.0
7,estudio_origen_posible,VARCHAR,no,si,2,None,None,None,None,None,14642142,0.0
8,estudio_destino_posible,VARCHAR,no,si,2,None,None,None,None,None,14642142,0.0
9,residencia,VARCHAR,01,52,53,None,None,None,None,None,14642142,0.0


viajes / municipios


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,9693500,0.0
1,periodo,VARCHAR,00,23,24,None,None,None,None,None,9693500,0.0
2,origen,VARCHAR,01001,externo,2696,None,None,None,None,None,9693500,0.0
3,destino,VARCHAR,01001,externo,2696,None,None,None,None,None,9693500,0.0
4,distancia,VARCHAR,0.5-2,>50,4,None,None,None,None,None,9693500,0.0
5,actividad_origen,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,9693500,0.0
6,actividad_destino,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,9693500,0.0
7,estudio_origen_posible,VARCHAR,no,si,2,None,None,None,None,None,9693500,0.0
8,estudio_destino_posible,VARCHAR,no,si,2,None,None,None,None,None,9693500,0.0
9,residencia,VARCHAR,01,52,53,None,None,None,None,None,9693500,0.0


viajes / GAU


...

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,119296,0.0
1,zona_pernoctacion,VARCHAR,01001,5200108,3818,None,None,None,None,None,119296,0.0
2,edad,VARCHAR,0-25,65-100,4,None,None,None,None,None,119296,0.0
3,sexo,VARCHAR,hombre,mujer,2,None,None,None,None,None,119296,0.0
4,numero_viajes,VARCHAR,0,2+,4,None,None,None,None,None,119296,0.0
5,personas,VARCHAR,10.000,999.979,106840,None,None,None,None,None,119296,0.0


personas / municipios


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,81805,0.0
1,zona_pernoctacion,VARCHAR,01001,52001,2594,None,None,None,None,None,81805,0.0
2,edad,VARCHAR,0-25,65-100,4,None,None,None,None,None,81805,0.0
3,sexo,VARCHAR,hombre,mujer,2,None,None,None,None,None,81805,0.0
4,numero_viajes,VARCHAR,0,2+,4,None,None,None,None,None,81805,0.0
5,personas,VARCHAR,10.0,9996.749,71487,None,None,None,None,None,81805,0.0


personas / GAU


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,64792,0.0
1,zona_pernoctacion,VARCHAR,01001,GAU Ávila,2063,None,None,None,None,None,64792,0.0
2,edad,VARCHAR,0-25,65-100,4,None,None,None,None,None,64792,0.0
3,sexo,VARCHAR,hombre,mujer,2,None,None,None,None,None,64792,0.0
4,numero_viajes,VARCHAR,0,2+,4,None,None,None,None,None,64792,0.0
5,personas,VARCHAR,10.067,999.944,57549,None,None,None,None,None,64792,0.0


In [5]:
# Una consulta por tabla: distinguimos vacíos, NULL y supresión por privacidad.
for familia in ['viajes', 'pernoctaciones', 'personas']:
    columnas = con.table(f'raw_{familia}').columns[2:]
    controles = []
    for columna in columnas:
        controles += [
            f"count(*) FILTER (WHERE {columna} = '') AS {columna}_vacios",
            f"count(*) FILTER (WHERE {columna} IS NULL) AS {columna}_null"
        ]
    print(familia)
    display(con.sql(f"SELECT nivel, {', '.join(controles)} FROM raw_{familia} GROUP BY nivel ORDER BY nivel").df().set_index('nivel').T)

display(con.sql("""
    SELECT nivel, count(*) AS filas,
           count(*) FILTER (WHERE edad = 'NA') AS edad_privada,
           count(*) FILTER (WHERE sexo = 'NA') AS sexo_privado,
           round(100.0 * sum(CASE WHEN edad = 'NA' OR sexo = 'NA'
                                THEN CAST(viajes AS DOUBLE) ELSE 0 END)
                 / sum(CAST(viajes AS DOUBLE)), 2) AS pct_viajes_con_privacidad
    FROM raw_viajes GROUP BY nivel ORDER BY nivel
""").df())

viajes


nivel,GAU,distritos,municipios
fecha_vacios,0,0,0
fecha_null,0,0,0
periodo_vacios,0,0,0
periodo_null,0,0,0
origen_vacios,0,0,0
origen_null,0,0,0
destino_vacios,0,0,0
destino_null,0,0,0
distancia_vacios,0,0,0
distancia_null,0,0,0


pernoctaciones


nivel,GAU,distritos,municipios
fecha_vacios,0,0,0
fecha_null,0,0,0
zona_residencia_vacios,0,0,0
zona_residencia_null,0,0,0
zona_pernoctacion_vacios,0,0,0
zona_pernoctacion_null,0,0,0
personas_vacios,0,0,0
personas_null,0,0,0


personas


nivel,GAU,distritos,municipios
fecha_vacios,0,0,0
fecha_null,0,0,0
zona_pernoctacion_vacios,0,0,0
zona_pernoctacion_null,0,0,0
edad_vacios,0,0,0
edad_null,0,0,0
sexo_vacios,0,0,0
sexo_null,0,0,0
numero_viajes_vacios,0,0,0
numero_viajes_null,0,0,0


,nivel,filas,edad_privada,sexo_privado,pct_viajes_con_privacidad
0,GAU,6222637,2175771,3494151,38.1
1,distritos,14642142,3778202,6480498,38.1
2,municipios,9693500,2872945,4935052,38.1


In [6]:
medidas = {'viajes': ['viajes', 'viajes_km'],
           'pernoctaciones': ['personas'], 'personas': ['personas']}
for familia, columnas in medidas.items():
    print(familia)
    display(con.sql(f"""
        SELECT nivel,
               count(*) FILTER (WHERE try_strptime(fecha, '%Y%m%d') IS NULL) AS fecha_invalida,
               count(*) FILTER (WHERE fecha <> regexp_extract(archivo_origen, '([0-9]{{8}})_', 1)) AS fecha_distinta_archivo
        FROM raw_{familia} GROUP BY nivel ORDER BY nivel
    """).df())
    for columna in columnas:
        display(con.sql(f"""
            SELECT nivel, '{columna}' AS medida,
                   count(*) FILTER (WHERE {columna} = '') AS ausentes,
                   count(*) FILTER (WHERE {columna} <> '' AND
                       (try_cast({columna} AS DOUBLE) IS NULL
                        OR NOT isfinite(try_cast({columna} AS DOUBLE)))) AS invalidos_no_vacios,
                   count(*) FILTER (WHERE try_cast({columna} AS DOUBLE) < 0) AS negativos,
                   count(*) FILTER (WHERE try_cast({columna} AS DOUBLE) = 0) AS ceros
            FROM raw_{familia} GROUP BY nivel ORDER BY nivel
        """).df())
    for nivel in ['distritos', 'municipios', 'GAU']:
        seleccion = ', '.join(f'TRY_CAST({c} AS DOUBLE) AS {c}' for c in columnas)
        print(nivel)
        display(con.sql(f"SUMMARIZE SELECT {seleccion} FROM raw_{familia} WHERE nivel = '{nivel}'").df())

viajes


,nivel,fecha_invalida,fecha_distinta_archivo
0,GAU,0,0
1,distritos,0,0
2,municipios,0,0


,nivel,medida,ausentes,invalidos_no_vacios,negativos,ceros
0,GAU,viajes,0,0,0,0
1,distritos,viajes,0,0,0,0
2,municipios,viajes,0,0,0,0


,nivel,medida,ausentes,invalidos_no_vacios,negativos,ceros
0,GAU,viajes_km,466,0,0,0
1,distritos,viajes_km,0,0,0,1397
2,municipios,viajes_km,640,0,0,0


distritos


...

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,personas,DOUBLE,10.0,16492.667,107144,396.46424418253963,680.4411170581283,101.63532348104941,202.8612467704194,428.2203689521955,119296,0.0


municipios


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,personas,DOUBLE,10.0,191189.294,72396,578.1626853370875,2681.220014593358,98.90712862036453,201.91473737487686,453.79642651010664,81805,0.0


GAU


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,personas,DOUBLE,10.067,369925.046,58731,729.9758994011593,6363.221404177784,90.41728793900887,174.19736126820538,378.64197731314243,64792,0.0


## 3. Categorías y granularidad
No homologamos edades: las etiquetas locales difieren del diccionario. `numero_viajes` es una categoría; `2+` no es el número 2. En viajes se suman medidas expandidas, no filas.

In [7]:
categorias = {'viajes': ['periodo', 'distancia', 'actividad_origen', 'actividad_destino',
                          'estudio_origen_posible', 'estudio_destino_posible',
                          'renta', 'edad', 'sexo'],
              'personas': ['edad', 'sexo', 'numero_viajes']}
for familia, columnas in categorias.items():
    consultas = [f"SELECT nivel, '{c}' AS campo, {c} AS valor, count(*) AS filas FROM raw_{familia} GROUP BY ALL"
                 for c in columnas]
    frecuencias = con.sql(' UNION ALL '.join(consultas)).df()
    print(familia)
    with pd.option_context('display.max_rows', None):
        display(frecuencias.pivot(index=['campo', 'valor'], columns='nivel', values='filas').fillna(0))

viajes


nivel                                        GAU  distritos  municipios
campo                   valor                                          
actividad_destino       casa             1995061    5288617     3277342
                        frecuente        1714768    4456334     2832706
                        no_frecuente     1897663    3502606     2624361
                        trabajo_estudio   615145    1394585      959091
actividad_origen        casa             1578155    4191569     2587797
                        frecuente        1728540    4568853     2869393
                        no_frecuente     2318423    4493741     3297726
                        trabajo_estudio   597519    1387979      938584
distancia               0.5-2            1146587    4155834     1963630
                        10-50            2071079    3329725     2855218
                        2-10             2493712    6553143     4309125
                        >50               511259     603440      565527
edad                    0-25              720555    1676112     1145240
                        25-45            1159056    3177917     1982904
                        45-65            1486138    4166129     2569188
                        65-100            681117    1843782     1123223
                        NA               2175771    3778202     2872945
estudio_destino_posible no               6184343   14570466     9641138
                        si                 38294      71676       52362
estudio_origen_posible  no               6186446   14574571     9643818
                        si                 36191      67571       49682
periodo                 00                271355     674180      435314
                        01                279733     720643      458439
                        02                219016     557533      359439
                        03                172660     422202      278713
                        04                148762     346858      235112
                        05                141251     322202      221521
                        06                162194     376968      253943
                        07                174682     403148      268257
                        08                175555     386848      263935
                        09                198952     431021      296546
                        10                238187     511564      353387
                        11                288366     621116      428700
                        12                341311     754528      517162
                        13                386203     908232      602836
                        14                353183     847019      552443
                        15                306065     696783      468579
                        16                346321     775940      526947
                        17                379754     872336      585400
                        18                384095     909956      598597
                        19                320495     788381      506490
                        20                297196     738325      470804
                        21                257271     643759      410093
                        22                218983     542556      347879
                        23                161047     390044      252964
renta                   10-15            3796606    9510089     6175290
                        <10              1539016    2583953     1937236
                        >15               887015    2548100     1580974
sexo                    NA               3494151    6480498     4935052
                        hombre           1411877    4249512     2471405
                        mujer            1316609    3912132     2287043

personas


nivel                   GAU  distritos  municipios
campo         valor                               
edad          0-25    16175      29793       20426
              25-45   16220      29856       20476
              45-65   16220      29856       20476
              65-100  16177      29791       20427
numero_viajes 0       16232      29869       20488
              1       16143      29735       20391
              2       16218      29854       20472
              2+      16199      29838       20454
sexo          hombre  32402      59653       40908
              mujer   32390      59643       40897

In [8]:
# Clave candidata: nivel + todas las dimensiones publicadas (sin archivo ni medidas).
for familia, columnas in medidas.items():
    excluir = ', '.join(['archivo_origen'] + columnas)
    print(familia)
    display(con.sql(f"""
        WITH grupos AS (
            SELECT * EXCLUDE ({excluir}), count(*) AS n
            FROM raw_{familia} GROUP BY ALL
        )
        SELECT nivel, count(*) AS grupos,
               count(*) FILTER (WHERE n > 1) AS claves_repetidas,
               sum(n - 1) AS filas_excedentes
        FROM grupos GROUP BY nivel ORDER BY nivel
    """).df())

viajes


,nivel,grupos,claves_repetidas,filas_excedentes
0,GAU,6222637,0,0.0
1,distritos,14641591,513,551.0
2,municipios,9693500,0,0.0


pernoctaciones


,nivel,grupos,claves_repetidas,filas_excedentes
0,GAU,434829,0,0.0
1,distritos,726907,0,0.0
2,municipios,560635,0,0.0


personas


,nivel,grupos,claves_repetidas,filas_excedentes
0,GAU,64792,0,0.0
1,distritos,119296,0,0.0
2,municipios,81805,0,0.0


## 4. Auxiliares y correspondencias
Población por zona no tiene cabecera. El índice exportado en nombres de municipios no es una clave. Comprobamos claves antes de unir nombres y población.

In [9]:
con.sql('CREATE TEMP TABLE nombres (nivel VARCHAR, codigo VARCHAR, nombre VARCHAR)')
con.sql('CREATE TEMP TABLE poblacion (nivel VARCHAR, codigo VARCHAR, poblacion VARCHAR)')
for nivel in ['distritos', 'municipios', 'GAU']:
    carpeta = DATA / 'zonificación' / f'zonificacion_{nivel}'
    sufijo = 'gaus' if nivel == 'GAU' else nivel
    leer(carpeta / f'nombres_{sufijo}.csv').create_view('entrada_nombres')
    leer(carpeta / f'poblacion_{sufijo}.csv', False).create_view('entrada_poblacion')
    con.sql(f"INSERT INTO nombres SELECT '{nivel}', ID, name FROM entrada_nombres")
    con.sql(f"INSERT INTO poblacion SELECT '{nivel}', column0, column1 FROM entrada_poblacion")

leer(DATA / 'zonificación' / 'relacion_ine_zonificacionMitma.csv').create_view('relacion')
leer(DATA / 'zonificación' / 'poblacion.csv').create_view('poblacion_auxiliar')
leer(DATA / 'calidad' / 'diarios' / '*.csv').create_view('descartados')
for tabla in ['nombres', 'poblacion', 'relacion', 'poblacion_auxiliar', 'descartados']:
    print(tabla)
    display(con.sql(f'SELECT * FROM {tabla} LIMIT 3').df())
    display(con.sql(f'SUMMARIZE {tabla}').df())

nombres


,nivel,codigo,nombre
0,distritos,01001,Alegría-Dulantzi
1,distritos,01002,Amurrio
2,distritos,01004_AM,Artziniega agregacion de municipios


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,nivel,VARCHAR,GAU,municipios,3,None,None,None,None,None,8496,0.0
1,codigo,VARCHAR,01001,GAU Ávila,4413,None,None,None,None,None,8496,0.0
2,nombre,VARCHAR,A Coruña,Úbeda,4085,None,None,None,None,None,8496,0.0


poblacion


,nivel,codigo,poblacion
0,distritos,01001,2925.0
1,distritos,01002,10307.0
2,distritos,01004_AM,3005.0


...

,distrito,municipio,provincia,poblacion
0,01001,01001,01,2925.0
1,01002,01002,01,10307.0
2,01004_AM,01004_AM,01,3005.0


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,distrito,VARCHAR,01001,5200108,3890,None,None,None,None,None,3792,0.0
1,municipio,VARCHAR,01001,52001,2682,None,None,None,None,None,3792,0.0
2,provincia,VARCHAR,01,53,54,None,None,None,None,None,3792,0.0
3,poblacion,VARCHAR,100128.0,NA,3274,None,None,None,None,None,3792,0.0


descartados


,fecha,zona
0,20230101,10032_AM
1,20230101,19086_AM
2,20230101,19103_AM


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,9,0.0
1,zona,VARCHAR,10032_AM,4801701,9,None,None,None,None,None,9,0.0


In [10]:
for tabla in ['nombres', 'poblacion']:
    print(tabla)
    display(con.sql(f"""
        SELECT nivel, count(*) AS filas, count(DISTINCT codigo) AS codigos,
               count(*) FILTER (WHERE codigo = '' OR codigo IS NULL) AS sin_codigo
        FROM {tabla} GROUP BY nivel ORDER BY nivel
    """).df())

display(con.sql("""
    SELECT coalesce(n.nivel, p.nivel) AS nivel,
           count(*) FILTER (WHERE n.codigo IS NULL) AS sin_nombre,
           count(*) FILTER (WHERE p.codigo IS NULL) AS sin_poblacion,
           count(*) FILTER (WHERE try_cast(p.poblacion AS DOUBLE) IS NULL
                             OR NOT isfinite(try_cast(p.poblacion AS DOUBLE))
                             OR try_cast(p.poblacion AS DOUBLE) < 0) AS poblacion_no_utilizable
    FROM nombres n FULL JOIN poblacion p USING (nivel, codigo)
    GROUP BY ALL ORDER BY nivel
""").df())

# Diagnóstico: no convierte esta coincidencia en una correspondencia semántica probada.
display(con.sql("""
    SELECT count(*) AS filas_auxiliares, count(DISTINCT distrito) AS distritos,
           count(*) FILTER (WHERE n.codigo IS NULL) AS sin_codigo_en_nombres_distritos,
           count(*) FILTER (WHERE try_cast(a.poblacion AS DOUBLE) IS NULL
                             OR NOT isfinite(try_cast(a.poblacion AS DOUBLE))
                             OR try_cast(a.poblacion AS DOUBLE) < 0) AS poblacion_no_utilizable
    FROM poblacion_auxiliar a
    LEFT JOIN nombres n ON n.nivel = 'distritos' AND n.codigo = a.distrito
""").df())

nombres


,nivel,filas,codigos,sin_codigo
0,GAU,2086,2086,0
1,distritos,3792,3792,0
2,municipios,2618,2618,0


poblacion


,nivel,filas,codigos,sin_codigo
0,GAU,2086,2086,0
1,distritos,3792,3792,0
2,municipios,2618,2618,0


,nivel,sin_nombre,sin_poblacion,poblacion_no_utilizable
0,GAU,0,0,49
1,distritos,0,0,49
2,municipios,0,0,49


,filas_auxiliares,distritos,sin_codigo_en_nombres_distritos,poblacion_no_utilizable
0,3792,3792,0,49


In [11]:
# Cobertura por rol, sin unir indicadores entre sí.
roles = {'viajes': ['origen', 'destino'],
         'pernoctaciones': ['zona_residencia', 'zona_pernoctacion'],
         'personas': ['zona_pernoctacion']}
cobertura = []
for familia, columnas in roles.items():
    for columna in columnas:
        cobertura.append(con.sql(f"""
            SELECT r.nivel, '{familia}.{columna}' AS rol,
                   count(*) FILTER (WHERE r.{columna} = 'externo') AS filas_externo,
                   count(*) FILTER (WHERE n.codigo IS NULL AND r.{columna} <> 'externo') AS filas_sin_catalogo,
                   count(DISTINCT r.{columna}) FILTER
                       (WHERE n.codigo IS NULL AND r.{columna} <> 'externo') AS codigos_sin_catalogo
            FROM raw_{familia} r LEFT JOIN nombres n
              ON n.nivel = r.nivel AND n.codigo = r.{columna}
            GROUP BY r.nivel
        """).df())
display(pd.concat(cobertura).sort_values(['rol', 'nivel']))

for columna in ['distrito_mitma', 'municipio_mitma', 'gau_mitma']:
    display(con.sql(f"""
        SELECT '{columna}' AS clave, count(*) AS filas_puente,
               count(DISTINCT {columna}) AS zonas,
               count(*) FILTER (WHERE {columna} = '' OR {columna} IS NULL) AS sin_zona,
               (SELECT max(n) FROM (SELECT count(*) AS n FROM relacion GROUP BY {columna})) AS max_filas_por_zona
        FROM relacion
    """).df())

display(con.sql("""
    SELECT count(*) AS filas, count(DISTINCT seccion_ine) AS secciones,
           count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM relacion)) AS duplicados_exactos
    FROM relacion
""").df())
for origen, destino in [('distrito_mitma', 'municipio_mitma'), ('municipio_mitma', 'gau_mitma')]:
    display(con.sql(f"""
        SELECT '{origen} -> {destino}' AS relacion,
               count(*) FILTER (WHERE destinos > 1) AS origenes_con_varios_destinos
        FROM (SELECT {origen}, count(DISTINCT {destino}) AS destinos FROM relacion GROUP BY {origen})
    """).df())

,nivel,rol,filas_externo,filas_sin_catalogo,codigos_sin_catalogo
0,GAU,pernoctaciones.zona_pernoctacion,0,0,0
2,distritos,pernoctaciones.zona_pernoctacion,0,0,0
1,municipios,pernoctaciones.zona_pernoctacion,0,0,0
0,GAU,pernoctaciones.zona_residencia,0,313108,1981
2,distritos,pernoctaciones.zona_residencia,0,0,0
1,municipios,pernoctaciones.zona_residencia,0,329762,1596
2,GAU,personas.zona_pernoctacion,0,0,0
0,distritos,personas.zona_pernoctacion,0,0,0
1,municipios,personas.zona_pernoctacion,0,0,0
1,GAU,viajes.destino,2498,51584,111


,clave,filas_puente,zonas,sin_zona,max_filas_por_zona
0,distrito_mitma,36382,3792,0,200


,clave,filas_puente,zonas,sin_zona,max_filas_por_zona
0,municipio_mitma,36382,2618,0,2443


,clave,filas_puente,zonas,sin_zona,max_filas_por_zona
0,gau_mitma,36382,2086,0,4141


,filas,secciones,duplicados_exactos
0,36382,36334,0


,relacion,origenes_con_varios_destinos
0,distrito_mitma -> municipio_mitma,0


,relacion,origenes_con_varios_destinos
0,municipio_mitma -> gau_mitma,0


In [12]:
display(con.sql("""
    SELECT count(*) AS filas,
           count(*) - count(DISTINCT (fecha, zona)) AS duplicados,
           count(*) FILTER (WHERE try_strptime(fecha, '%Y%m%d') IS NULL) AS fecha_invalida,
           count(*) FILTER (WHERE n.codigo IS NULL) AS sin_catalogo
    FROM descartados d LEFT JOIN nombres n ON n.nivel = 'distritos' AND n.codigo = d.zona
""").df())

display(con.sql("""
    SELECT d.fecha, d.zona,
           (SELECT count(*) FROM raw_viajes v WHERE v.nivel = 'distritos'
              AND v.fecha = d.fecha AND (v.origen = d.zona OR v.destino = d.zona)) AS filas_viajes
    FROM descartados d ORDER BY d.fecha, d.zona
""").df())

,filas,duplicados,fecha_invalida,sin_catalogo
0,9,0,0,0


,fecha,zona,filas_viajes
0,20230101,10032_AM,0
1,20230101,19086_AM,0
2,20230101,19103_AM,0
3,20230101,19156_AM,0
4,20230101,25158_AM,0
5,20230101,31260_AM,0
6,20230101,31905_AM,0
7,20230101,40149_AM,0
8,20230101,4801701,0


### Comprobación de ausencias del puente
`NA` también aparece en códigos INE; aquí no debe asumirse la misma causa que en edad/sexo. No sirve como clave de sección.

In [13]:
display(con.sql("""
    SELECT count(*) FILTER (WHERE seccion_ine = 'NA') AS secciones_na,
           count(*) FILTER (WHERE seccion_ine <> 'NA') AS filas_con_seccion,
           count(DISTINCT seccion_ine) FILTER (WHERE seccion_ine <> 'NA') AS secciones_identificadas
    FROM relacion
""").df())

display(con.sql("""
    SELECT count(*) AS filas_comparadas,
           count(*) FILTER (WHERE p.codigo IS NULL) AS sin_poblacion_zona,
           count(*) FILTER (WHERE a.poblacion <> p.poblacion) AS poblacion_distinta
    FROM poblacion_auxiliar a LEFT JOIN poblacion p
      ON p.nivel = 'distritos' AND p.codigo = a.distrito
""").df())

display(con.sql("SELECT * FROM raw_viajes WHERE viajes_km = '' LIMIT 3").df())

,secciones_na,filas_con_seccion,secciones_identificadas
0,49,36333,36333


,filas_comparadas,sin_poblacion_zona,poblacion_distinta
0,3792,0,0


,nivel,archivo_origen,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
0,GAU,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,20230101,00,02009,02009,0.5-2,no_frecuente,no_frecuente,no,no,03,<10,45-65,NA,1.0,
1,GAU,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,20230101,00,03015,03015,0.5-2,no_frecuente,frecuente,no,no,03,<10,NA,NA,1.0,
2,GAU,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,20230101,00,03059,03059,0.5-2,frecuente,no_frecuente,no,no,03,<10,NA,NA,1.0,


## 5. Prototipo Silver y reconciliación
`viajes_km` tiene 640 vacíos en municipios y 466 en GAU: se convierten explícitamente en NULL con `viajes_km_ausente`, sin imputación. Las sumas representan la parte conocida. Vistas en memoria, sin filtrar filas ni unir hechos con auxiliares.

In [14]:
for familia, columnas in medidas.items():
    conversiones = ["CAST(strptime(fecha, '%Y%m%d') AS DATE) AS fecha"]
    for columna in columnas:
        # Solo viajes_km tiene vacíos observados: NULL explícito, nunca cero.
        valor = f"nullif({columna}, '')" if columna == 'viajes_km' else columna
        conversiones.append(f'CAST({valor} AS DOUBLE) AS {columna}')
    marca = ", viajes_km = '' AS viajes_km_ausente" if familia == 'viajes' else ''
    con.sql(f"CREATE TEMP VIEW silver_{familia} AS SELECT * REPLACE ({', '.join(conversiones)}) {marca} FROM raw_{familia}")
    print(f'silver_{familia}')
    display(con.sql(f'DESCRIBE silver_{familia}').df())
    for columna in columnas:
        comparacion = con.sql(f"""
            WITH bronze AS (
                SELECT nivel, count(*) AS filas_bronze,
                       count(*) FILTER (WHERE {columna} = '') AS ausentes_bronze,
                       sum(CAST(nullif({columna}, '') AS DOUBLE)) AS total_bronze
                FROM raw_{familia} GROUP BY nivel
            ), silver AS (
                SELECT nivel, count(*) AS filas_silver,
                       count(*) FILTER (WHERE {columna} IS NULL) AS ausentes_silver,
                       sum({columna}) AS total_silver
                FROM silver_{familia} GROUP BY nivel
            )
            SELECT *, filas_bronze = filas_silver AS mismas_filas,
                   ausentes_bronze = ausentes_silver AS mismos_ausentes,
                   abs(total_bronze - total_silver) <= 1e-6 + abs(total_bronze) * 1e-12 AS mismo_total
            FROM bronze JOIN silver USING (nivel) ORDER BY nivel
        """).df()
        print(columna)
        display(comparacion)
        assert comparacion[['mismas_filas', 'mismos_ausentes', 'mismo_total']].all().all()

# La marca debe coincidir con el NULL generado desde el vacío original.
assert con.sql("SELECT count(*) FROM silver_viajes WHERE viajes_km_ausente <> (viajes_km IS NULL)").fetchall()[0][0] == 0

silver_viajes


,column_name,column_type,null,key,default,extra
0,nivel,VARCHAR,YES,None,None,None
1,archivo_origen,VARCHAR,YES,None,None,None
2,fecha,DATE,YES,None,None,None
3,periodo,VARCHAR,YES,None,None,None
4,origen,VARCHAR,YES,None,None,None
5,destino,VARCHAR,YES,None,None,None
6,distancia,VARCHAR,YES,None,None,None
7,actividad_origen,VARCHAR,YES,None,None,None
8,actividad_destino,VARCHAR,YES,None,None,None
9,estudio_origen_posible,VARCHAR,YES,None,None,None


viajes


,nivel,filas_bronze,ausentes_bronze,total_bronze,filas_silver,ausentes_silver,total_silver,mismas_filas,mismos_ausentes,mismo_total
0,GAU,6222637,0,9.395200e+07,6222637,0,9.395200e+07,True,True,True
1,distritos,14642142,0,9.395200e+07,14642142,0,9.395200e+07,True,True,True
2,municipios,9693500,0,9.395200e+07,9693500,0,9.395200e+07,True,True,True


viajes_km


...

,nivel,filas_bronze,ausentes_bronze,total_bronze,filas_silver,ausentes_silver,total_silver,mismas_filas,mismos_ausentes,mismo_total
0,GAU,434829,0,4.729773e+07,434829,0,4.729773e+07,True,True,True
1,distritos,726907,0,4.729773e+07,726907,0,4.729773e+07,True,True,True
2,municipios,560635,0,4.729773e+07,560635,0,4.729773e+07,True,True,True


silver_personas


,column_name,column_type,null,key,default,extra
0,nivel,VARCHAR,YES,None,None,None
1,archivo_origen,VARCHAR,YES,None,None,None
2,fecha,DATE,YES,None,None,None
3,zona_pernoctacion,VARCHAR,YES,None,None,None
4,edad,VARCHAR,YES,None,None,None
5,sexo,VARCHAR,YES,None,None,None
6,numero_viajes,VARCHAR,YES,None,None,None
7,personas,DOUBLE,YES,None,None,None


personas


,nivel,filas_bronze,ausentes_bronze,total_bronze,filas_silver,ausentes_silver,total_silver,mismas_filas,mismos_ausentes,mismo_total
0,GAU,64792,0,4.729660e+07,64792,0,4.729660e+07,True,True,True
1,distritos,119296,0,4.729660e+07,119296,0,4.729660e+07,True,True,True
2,municipios,81805,0,4.729660e+07,81805,0,4.729660e+07,True,True,True


## 6. Casos que requieren cautela
Hay 513 claves de viajes repetidas en distritos (551 filas excedentes). Comprobamos si son también duplicados exactos. La población `NA` se representa como NULL conservando el texto original; las zonas sin catálogo se muestran, no se eliminan.

In [15]:
display(con.sql("""
    SELECT count(*) AS grupos_exactos_repetidos, coalesce(sum(n - 1), 0) AS filas_excedentes
    FROM (SELECT * EXCLUDE (archivo_origen), count(*) AS n
          FROM raw_viajes WHERE nivel = 'distritos' GROUP BY ALL HAVING count(*) > 1)
""").df())

# Ejemplos de referencias ausentes; no se presupone su equivalencia con otro nivel.
for familia, columna in [('viajes', 'origen'), ('pernoctaciones', 'zona_residencia')]:
    print(familia, columna)
    display(con.sql(f"""
        SELECT DISTINCT r.nivel, r.{columna} AS codigo
        FROM raw_{familia} r LEFT JOIN nombres n
          ON n.nivel = r.nivel AND n.codigo = r.{columna}
        WHERE n.codigo IS NULL AND r.{columna} <> 'externo'
        ORDER BY r.nivel, codigo LIMIT 8
    """).df())

display(con.sql("SELECT nivel, poblacion, count(*) AS filas FROM poblacion WHERE try_cast(poblacion AS DOUBLE) IS NULL GROUP BY ALL ORDER BY nivel").df())

con.sql("""
    CREATE TEMP VIEW silver_zona AS
    SELECT nivel, codigo, nombre, p.poblacion AS poblacion_original,
           CAST(nullif(p.poblacion, 'NA') AS DOUBLE) AS poblacion,
           p.poblacion = 'NA' AS poblacion_ausente
    FROM nombres n FULL JOIN poblacion p USING (nivel, codigo)
""")
display(con.sql("SELECT nivel, count(*) AS zonas, count(*) FILTER (WHERE poblacion IS NULL) AS sin_poblacion FROM silver_zona GROUP BY nivel ORDER BY nivel").df())

,grupos_exactos_repetidos,filas_excedentes
0,1,1.0


viajes origen


,nivel,codigo
0,GAU,FR101
1,GAU,FR102
2,GAU,FR103
3,GAU,FR104
4,GAU,FR105
5,GAU,FR106
6,GAU,FR107
7,GAU,FR108


pernoctaciones zona_residencia


,nivel,codigo
0,GAU,0105901
1,GAU,0105902
2,GAU,0105903
3,GAU,0105904
4,GAU,0105905
5,GAU,0105906
6,GAU,0200301
7,GAU,0200302


,nivel,poblacion,filas
0,GAU,NA,49
1,distritos,NA,49
2,municipios,NA,49


,nivel,zonas,sin_poblacion
0,GAU,2086,49
1,distritos,3792,49
2,municipios,2618,49


## 7. Decisiones
Solo una repetición es exacta; las demás repiten dimensiones con medidas distintas. Entre los códigos sin catálogo hay `FR101` y otros territorios fuera del catálogo nacional; no basta con tratar el literal `externo`.

| Hallazgo | Decisión Silver |
|---|---|
| Códigos con ceros iniciales y tres zonificaciones | Texto y clave territorial `(nivel, codigo)`. |
| El 38,10 % de viajes tiene edad o sexo `NA` | Conservar supresión, sin perder medidas. |
| 640/466 viajes-km vacíos (municipios/GAU) | NULL con marca; totales de kilómetros solo conocidos. |
| Claves repetidas en viajes de distritos | Sin PK de segmento impuesta ni deduplicación automática. |
| 49 poblaciones `NA` por nivel | NULL con texto original y marca de ausencia. |
| Referencias fuera del catálogo | Relaciones opcionales y revisión; no descartar hechos. |
| Puente con varias filas por zona y secciones `NA` | Validar pares antes del join; `NA` no es clave de sección. |

Contrato completo en [modelo Bronze/Silver](../docs/modelo_bronze_silver_mitma.md). Las claves y coberturas deben volver a comprobarse en cada carga; un solo día no permite estudiar evolución temporal.